# BÀI TẬP LỚN SỐ 1: MULTIMODAL CLASSIFICATION
* **Môn học:** Học sâu và ứng dụng trong thị giác máy tính
* **Nhóm:** group6
* **Thành viên**:

## Chuẩn bị môi trường

In [1]:
!pip install transformers torch scikit-learn pillow tqdm

In [2]:
import os
import pandas as pd
import numpy as np
import torch
from PIL import Image
from transformers import CLIPProcessor, CLIPModel
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings("ignore")

In [3]:
DATA_DIR = "/kaggle/input/datasets/hsankesara/flickr-image-dataset/flickr30k_images"
IMAGES_DIR = os.path.join(DATA_DIR, "flickr30k_images")
CAPTIONS_FILE = os.path.join(DATA_DIR, "results.csv")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

TARGET_CLASSES = ["person", "dog", "car", "bus", "bike"]

cuda


## Tải mô hình CLIP

In [4]:
model_name = "openai/clip-vit-base-patch32"
model = CLIPModel.from_pretrained(model_name).to(device)
processor = CLIPProcessor.from_pretrained(model_name)
print("Đã tải xong mô hình CLIP")

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

Đã tải xong mô hình CLIP


## Tiền xử lý dữ liệu

In [5]:
print("Đang đọc và xử lý dữ liệu...")

df_captions = pd.read_csv(CAPTIONS_FILE, sep='|')

# QUAN TRỌNG: Loại bỏ khoảng trắng thừa ở tên cột
df_captions.columns = df_captions.columns.str.strip()

df_captions.rename(columns={'image_name': 'image'}, inplace=True)

if 'comment' not in df_captions.columns:
    # Tìm cột chứa chữ 'comment' gần nhất
    comment_col = [c for c in df_captions.columns if 'comment' in c.lower()]
    if comment_col:
        df_captions.rename(columns={comment_col[0]: 'comment'}, inplace=True)

# Gom nhóm captions
grouped = df_captions.groupby('image')['comment'].apply(list).reset_index()
grouped.columns = ['image', 'captions']

print(f"Các cột sau khi làm sạch: {df_captions.columns.tolist()}")

Đang đọc và xử lý dữ liệu...
Các cột sau khi làm sạch: ['image', 'comment_number', 'comment']


In [6]:
# Lọc ảnh tồn tại
valid_rows = []
for idx, row in grouped.iterrows():
    img_path = os.path.join(IMAGES_DIR, row['image'])
    if os.path.exists(img_path):
        valid_rows.append({
            'path': img_path,
            'captions': row['captions'],
            'image_name': row['image']
        })


In [7]:
df_valid = pd.DataFrame(valid_rows)
print(f"Tổng số ảnh hợp lệ: {len(df_valid)}")
stop_execution = False
# Hàm gán nhãn (Cải thiện độ nhạy)
def assign_label(captions_list):
    counts = {cls: 0 for cls in TARGET_CLASSES}
    for cap in captions_list:
        if not isinstance(cap, str): continue
        cap_lower = cap.lower()
        for cls in TARGET_CLASSES:
            # Kiểm tra từ khóa, bao gồm cả số nhiều đơn giản
            if f" {cls} " in f" {cap_lower} " or f" {cls}s " in f" {cap_lower} " or cap_lower.endswith(f" {cls}"):
                counts[cls] += 1
            elif cls in cap_lower: # Fallback nếu từ nằm giữa câu không có khoảng trắng chuẩn
                 counts[cls] += 0.5 # Giảm trọng số nếu match không chắc chắn
    
    max_cls = max(counts, key=counts.get)
    if counts[max_cls] == 0:
        return None
    return max_cls

df_valid['label'] = df_valid['captions'].apply(assign_label)
df_clean = df_valid.dropna(subset=['label']).reset_index(drop=True)

print(f"Số lượng ảnh sau khi lọc theo nhãn {TARGET_CLASSES}: {len(df_clean)}")
if len(df_clean) > 0:
    print(df_clean['label'].value_counts())
else:
    print("Cảnh báo: Không tìm thấy ảnh nào khớp với các lớp mục tiêu. Hãy kiểm tra lại từ khóa hoặc dữ liệu.")

Tổng số ảnh hợp lệ: 31783
Số lượng ảnh sau khi lọc theo nhãn ['person', 'dog', 'car', 'bus', 'bike']: 10599
label
car       3108
person    2722
dog       2364
bike      1215
bus       1190
Name: count, dtype: int64


In [8]:
print("\n--- Bắt đầu Zero-Shot Classification ---")

templates = [
    "a photo of a {}",
    "a picture of a {}",
    "an image of a {}",
    "{}",
    "a blurry photo of a {}"
]

zero_shot_preds = []
true_labels = []


with torch.no_grad():
    for idx, row in tqdm(df_clean.iterrows(), total=len(df_clean), desc="Zero-Shot"):
        img_path = row['path']
        true_label = row['label']
        true_labels.append(true_label)
        
        try:
            image = Image.open(img_path).convert("RGB")
        except Exception:
            continue

        texts = []
        for cls in TARGET_CLASSES:
            for tmpl in templates:
                texts.append(tmpl.format(cls))
        
        inputs = processor(text=texts, images=image, return_tensors="pt", padding=True).to(device)
        
        # CÁCH GỌI AN TOÀN NHẤT: Gọi model trực tiếp và lấy embeds
        outputs = model(**inputs)
        
        # Trích xuất trực tiếp từ trường image_embeds và text_embeds
        image_features = outputs.image_embeds
        text_features = outputs.text_embeds
        
        # Debug type (sẽ thấy là torch.Tensor)
        # if idx == 0: print(f"Type: {type(image_features)}")

        # Chuẩn hóa
        eps = 1e-8
        image_features = image_features / (image_features.norm(dim=-1, keepdim=True) + eps)
        text_features = text_features / (text_features.norm(dim=-1, keepdim=True) + eps)
        
        # Tính similarity
        logits_per_image = image_features @ text_features.T
        probs = logits_per_image.softmax(dim=-1).cpu().numpy()[0]
        
        # Gộp điểm
        class_scores = []
        num_templates = len(templates)
        for i, cls in enumerate(TARGET_CLASSES):
            start = i * num_templates
            end = start + num_templates
            avg_score = np.mean(probs[start:end])
            class_scores.append(avg_score)
        
        pred_class = TARGET_CLASSES[np.argmax(class_scores)]
        zero_shot_preds.append(pred_class)

zs_accuracy = accuracy_score(true_labels, zero_shot_preds)
print(f"\nZero-Shot Accuracy: {zs_accuracy:.4f}")
if len(set(true_labels)) > 1:
    print(classification_report(true_labels, zero_shot_preds, target_names=TARGET_CLASSES))
else:
    print("Không thể hiện classification report do chỉ có 1 lớp trong tập test.")


--- Bắt đầu Zero-Shot Classification ---


Zero-Shot:   0%|          | 0/10599 [00:00<?, ?it/s]


Zero-Shot Accuracy: 0.5941
              precision    recall  f1-score   support

      person       0.67      0.91      0.77      1215
         dog       0.45      0.20      0.28      1190
         car       0.83      0.16      0.27      3108
         bus       0.97      0.93      0.95      2364
        bike       0.41      0.84      0.55      2722

    accuracy                           0.59     10599
   macro avg       0.67      0.61      0.56     10599
weighted avg       0.69      0.59      0.55     10599



In [10]:
print("\n--- Bắt đầu Few-Shot Classification ---")

shots_per_class = 128
train_data = []
test_data = []

for cls in TARGET_CLASSES:
    cls_df = df_clean[df_clean['label'] == cls]
    # Giảm điều kiện xuống còn 5 mẫu để tránh bỏ qua lớp nếu dữ liệu ít
    if len(cls_df) < shots_per_class + 5: 
        print(f"Bỏ qua lớp {cls} do thiếu dữ liệu (có {len(cls_df)} mẫu).")
        continue
    
    cls_df_shuffled = cls_df.sample(frac=1, random_state=42).reset_index(drop=True)
    train_chunk = cls_df_shuffled.iloc[:shots_per_class]
    test_chunk = cls_df_shuffled.iloc[shots_per_class:]
    
    train_data.append(train_chunk)
    test_data.append(test_chunk)

if len(train_data) == 0:
    print("LỖI: Không đủ dữ liệu để chạy Few-Shot (cần ít nhất 2 lớp có đủ mẫu).")
    print("Dừng chương trình tại đây.")
else:
    df_train = pd.concat(train_data, ignore_index=True)
    df_test = pd.concat(test_data, ignore_index=True)

    print(f"Train size: {len(df_train)}, Test size: {len(df_test)}")

    # Định nghĩa eps toàn cục cho hàm này để tránh lỗi NameError
    # Định nghĩa eps
    eps = 1e-8 

    def extract_embeddings(df_subset):
        embeddings = []
        labels = []
        success_count = 0
        
        for idx, row in tqdm(df_subset.iterrows(), total=len(df_subset), desc="Embedding"):
            try:
                img_path = row['path']
                if not os.path.exists(img_path):
                    continue
                    
                image = Image.open(img_path).convert("RGB")
                
                # CHỈ PROCESS ẢNH, KHÔNG CẦN TEXT
                inputs = processor(images=image, return_tensors="pt").to(device)
                
                with torch.no_grad():
                    # CÁCH SỬA ĐÚNG:
                    # Gọi trực tiếp vision_model để lấy hidden states, sau đó chiếu qua projection
                    # Hoặc đơn giản hơn: dùng model.get_image_features chỉ với pixel_values
                    
                    # Phương án an toàn nhất cho mọi version transformers:
                    vision_outputs = model.vision_model(pixel_values=inputs['pixel_values'])
                    pooler_output = vision_outputs.pooler_output # Lấy vector đã gộp
                    image_embeds = model.visual_projection(pooler_output) # Chiếu lên không gian chung
                    
                    img_emb = image_embeds
                    
                    # Chuẩn hóa
                    img_emb = img_emb / (img_emb.norm(dim=-1, keepdim=True) + eps)
                
                embeddings.append(img_emb.cpu().numpy())
                labels.append(row['label'])
                success_count += 1
            except Exception as e:
                # In lỗi của mẫu đầu tiên để debug
                if idx == 0:
                    print(f"Lỗi chi tiết tại ảnh đầu tiên: {e}")
                    print(f"Đường dẫn ảnh: {row['path']}")
                continue
        
        if len(embeddings) == 0:
            raise ValueError(f"Không trích xuất được embedding nào từ {len(df_subset)} ảnh.")
        
        print(f"Đã trích xuất thành công {success_count}/{len(df_subset)} ảnh.")
        return np.vstack(embeddings), labels

    print("Trích xuất Train Embeddings...")
    try:
        X_train, y_train = extract_embeddings(df_train)
        y_train_idx = [TARGET_CLASSES.index(l) for l in y_train]

        print("Trích xuất Test Embeddings...")
        X_test, y_test = extract_embeddings(df_test)
        y_test_idx = [TARGET_CLASSES.index(l) for l in y_test]

        clf = LogisticRegression(max_iter=1000, multi_class='multinomial', solver='lbfgs')
        clf.fit(X_train, y_train_idx)

        y_pred_idx = clf.predict(X_test)
        y_pred = [TARGET_CLASSES[i] for i in y_pred_idx]
        y_test_true = [TARGET_CLASSES[i] for i in y_test_idx]

        fs_accuracy = accuracy_score(y_test_true, y_pred)
        print(f"\nFew-Shot ({shots_per_class} shots) Accuracy: {fs_accuracy:.4f}")
        
        if len(set(y_test_true)) > 1:
            print(classification_report(y_test_true, y_pred, target_names=TARGET_CLASSES))
        else:
            print("Chỉ có 1 lớp trong tập test.")

        print("\n=== KẾT LUẬN CUỐI CÙNG ===")
        print(f"Zero-Shot Accuracy: {zs_accuracy:.4f}")
        print(f"Few-Shot Accuracy:  {fs_accuracy:.4f}")
        
        if fs_accuracy > zs_accuracy:
            print("=> Few-shot learning đã cải thiện hiệu suất!")
        else:
            print("=> Zero-shot vẫn hoạt động tốt hơn hoặc tương đương.")
            
    except ValueError as ve:
        print(f"\nLỗi nghiêm trọng: {ve}")


--- Bắt đầu Few-Shot Classification ---
Train size: 640, Test size: 9959
Trích xuất Train Embeddings...


Embedding:   0%|          | 0/640 [00:00<?, ?it/s]

Đã trích xuất thành công 640/640 ảnh.
Trích xuất Test Embeddings...


Embedding:   0%|          | 0/9959 [00:00<?, ?it/s]

Đã trích xuất thành công 9959/9959 ảnh.

Few-Shot (128 shots) Accuracy: 0.6590
              precision    recall  f1-score   support

      person       0.74      0.83      0.78      1087
         dog       0.34      0.69      0.45      1062
         car       0.65      0.49      0.56      2980
         bus       0.97      0.89      0.93      2236
        bike       0.65      0.57      0.60      2594

    accuracy                           0.66      9959
   macro avg       0.67      0.69      0.67      9959
weighted avg       0.70      0.66      0.67      9959


=== KẾT LUẬN CUỐI CÙNG ===
Zero-Shot Accuracy: 0.5941
Few-Shot Accuracy:  0.6590
=> Few-shot learning đã cải thiện hiệu suất!
